# 03 · Walking access and facility siting

**Question:** Which households are poorly served once streets and barriers are included?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
All city geometry, residents, sensor observations and parameters are **synthetic educational fixtures**.
The longitude/latitude anchor is near Gaborone, but these are not mapped Gaborone assets or a validated city twin.

**Astra experiment:** Ask Astra to choose a new clinic using the exported candidate scores, then check its answer against exhaustive search.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## A barrier makes distance a network problem
The vertical barrier has one northern crossing. Building-to-node connectors are straight-line approximations.
All streets are bidirectional; there is no timetable, capacity or clinical service model.


In [ ]:
c=city(); pop=allocate(c['capacity'],1200)
barrier=[(r*11+5,r*11+6) for r in range(10)]
nodes,graph=network(blocked=barrier)
clinic=22; speed_m_min=80.; threshold_min=15.
idx,connector=snap(c['xy'],nodes)
base=(shortest(graph,[clinic])[idx]+connector)/speed_m_min
euclidean=np.linalg.norm(c['xy']-nodes[clinic],axis=1)/speed_m_min
assert (base+1e-9>=euclidean).all()
scores=[]
for candidate in range(len(nodes)):
    times=(shortest(graph,[clinic,candidate])[idx]+connector)/speed_m_min
    scores.append(float(np.average(times,weights=pop)))
best=int(np.argmin(scores))
improved=(shortest(graph,[clinic,best])[idx]+connector)/speed_m_min
assert (improved<=base+1e-10).all()
print('Best new node:',best,'| weighted minutes:',np.average(base,weights=pop),'->',scores[best])
print('Residents within threshold:',pop[base<=threshold_min].sum(),'->',pop[improved<=threshold_min].sum())


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,times,label in zip(axes,[base,improved],['Baseline','With one additional clinic']):
    for u,edges in graph.items():
        for v,w in edges:
            if u<v: ax.plot(*nodes[[u,v]].T,color='#bcc8ce',lw=.6)
    dots=ax.scatter(*c['xy'].T,c=times,vmin=0,vmax=base.max(),cmap='magma_r',s=20)
    ax.scatter(*nodes[clinic],marker='P',c='cyan',s=140)
    if label!='Baseline': ax.scatter(*nodes[best],marker='*',c='lime',s=160)
    map_axes(ax,label)
fig.colorbar(dots,ax=list(axes),label='Walking minutes'); plt.show()
export_json('03_access.json',dict(synthetic=True,best_node=best,candidate_weighted_minutes=scores,threshold_min=threshold_min))


## Extend with Codex or Astra
Introduce disconnected components, wheelchair accessibility, clinic opening hours, service capacity and combi waiting times. Report unreachable residents separately from average travel time.

Keep the baseline seed and tests fixed while changing one assumption. Record the model name,
date, exact prompt, response and error metrics. Generated code must pass the same checks.
Download files in `exports/` and your edited notebook; browser storage does not commit changes to GitHub.
See [the project guide](https://github.com/jltobias/JupyterLite-GPT-6-Astra-Geospatial/blob/main/docs/GUIDE.md)
for capabilities, evidence, sources and the route to real data.
